# Hacking Forums — 01: Ingeniería de datos

En este notebook transformamos los dumps SQL crudos en un corpus limpio y estandarizado  
que pueden consumir todos los análisis posteriores.

El trabajo de ingeniería de datos es silencioso y poco glamoroso, pero es **el paso más crítico**:  
un corpus sucio produce resultados incorrectos que parecen correctos.  
Timestamps con fecha 1970, posts vacíos, y userids duplicados entre foros son  
exactamente el tipo de problema que destruye un análisis si no se detectan aquí.

**Output de este notebook**: dos archivos Parquet (formato columnar eficiente para DataFrames grandes):
- `posts_clean.parquet` — todos los posts limpios, con columna `forum` para identificar el origen
- `users_clean.parquet` — todos los usuarios limpios, misma estructura

## Importaciones y configuración

Cargamos las bibliotecas que vamos a usar en este notebook:
- `pandas`: la biblioteca principal para trabajar con tablas de datos (DataFrames)
- `matplotlib` y `seaborn`: para visualizaciones
- `load_forum`: la función del proyecto que parsea cada dump SQL

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.utils import load_forum, RESULTS_DIR, DATA_DIR

plt.style.use('dark_background')
sns.set_palette('muted')

HF_DIR     = DATA_DIR / 'Hacking Forums'
HF_RESULTS = RESULTS_DIR / 'hacking_forums'
HF_RESULTS.mkdir(parents=True, exist_ok=True)

FORUMS_WITH_POSTS = [
    "OGUsers_2019.zip",
    "Exploit.in_2013.12.13.zip",
    "Cracked.to_2019.01.zip",
    "Nulled.io_2016.05.zip",
    "RaidForums_2021.zip",
]

print(f"DATA_DIR:    {DATA_DIR}")
print(f"HF_RESULTS:  {HF_RESULTS}")
print(f"Foros objetivo: {len(FORUMS_WITH_POSTS)}")

## Sección 1 — Carga con load_forum()

### ¿Qué hace load_forum()?

`load_forum(path)` recibe la ruta a un archivo ZIP que contiene el dump SQL de un foro  
y devuelve un **diccionario de DataFrames**: `{'user': ..., 'post': ..., 'thread': ...}`.

La función hace dos cosas automáticamente:

1. **Auto-detección de versión**: mira el schema del dump para decidir si es MyBB, IPS 3.x con  
   prefijo `ibf_`, o IPS 3.x sin prefijo. Cada uno tiene nombres de tablas y columnas distintos.

2. **Normalización de columnas**: sin importar qué formato usaba el foro original,  
   la función devuelve siempre las mismas columnas: `userid`, `username`, `postid`, `threadid`,  
   `dateline`, `pagetext`. Esto es lo que permite comparar foros entre sí.

### Nota sobre Nulled.io

Nulled.io es IPS 3.x sin prefijo `ibf_`, lo cual hace que `is_mybb()` devuelva un falso positivo.  
Si el parser falla en Nulled.io, el código intenta forzar el modo IPS-sin-prefijo explícitamente.

### ¿Qué es el encoding?

El encoding define cómo se representan los caracteres en el archivo. Los dumps más viejos (2013–2016)  
usan frecuentemente `latin-1` o `cp1251` (para ruso). Si hay caracteres corruptos (`â€™` en lugar de `'`),  
es un problema de encoding. `load_forum()` intenta detectarlo automáticamente.

> `load_forum()` está definida en `src/utils.py` — es el punto de entrada que usa todo el curso para cargar dumps, no solo este caso. Si vas a procesar un dump propio, reutilízala en vez de llamar a `src/parsers/` directamente. Detalle completo de la API en `src/README.md`.


In [ ]:
raw_posts  = []
raw_users  = []
posts_by_forum = {}
users_by_forum = {}

for fname in FORUMS_WITH_POSTS:
    zip_path = HF_DIR / fname
    if not zip_path.exists():
        print(f"  SKIP (no encontrado): {fname}")
        continue
    try:
        dfs = load_forum(zip_path)
    except Exception as e:
        print(f"  ERROR {fname}: {e}")
        continue

    forum_label = Path(fname).stem

    if 'post' in dfs and len(dfs['post']) > 0:
        posts_df = dfs['post'].copy()
        posts_df['forum'] = forum_label
        for col in ['userid', 'postid', 'threadid']:
            if col in posts_df.columns:
                posts_df[col] = posts_df[col].astype(str)
        for col in ['postid', 'threadid', 'userid', 'username', 'dateline', 'pagetext']:
            if col not in posts_df.columns:
                posts_df[col] = None
        posts_df = posts_df[['forum', 'postid', 'threadid', 'userid', 'username', 'dateline', 'pagetext']]
        raw_posts.append(posts_df)
        posts_by_forum[fname] = posts_df
        print(f"  posts  {forum_label}: {len(posts_df):,} filas")
    else:
        print(f"  posts  {forum_label}: sin datos")

    if 'user' in dfs and len(dfs['user']) > 0:
        users_df = dfs['user'].copy()
        users_df['forum'] = forum_label
        if 'userid' in users_df.columns:
            users_df['userid'] = users_df['userid'].astype(str)
        for col in ['userid', 'username', 'email', 'joindate', 'ipaddress']:
            if col not in users_df.columns:
                users_df[col] = None
        users_df = users_df[['forum', 'userid', 'username', 'email', 'joindate', 'ipaddress']]
        raw_users.append(users_df)
        users_by_forum[fname] = users_df
        print(f"  users  {forum_label}: {len(users_df):,} filas")
    else:
        print(f"  users  {forum_label}: sin datos")

posts_raw = pd.concat(raw_posts, ignore_index=True) if raw_posts else pd.DataFrame()
users_raw = pd.concat(raw_users, ignore_index=True) if raw_users else pd.DataFrame()

print(f"\nPosts totales (raw):    {len(posts_raw):,}")
print(f"Usuarios totales (raw): {len(users_raw):,}")

## Sección 2 — Limpieza de posts

Aplicamos cuatro filtros en secuencia (texto vacío, duplicados por `(forum, postid)`, timestamps epoch-0 y texto demasiado corto) — el mismo patrón de limpieza ya visto en CardingForums, así que no lo repetimos en detalle aquí; el foco de este caso está en la identidad y la persistencia temporal, no en la ingeniería de datos.


In [ ]:
if len(posts_raw) == 0:
    print("Sin posts para limpiar.")
    posts_clean = posts_raw.copy()
else:
    counts_before = {forum: (posts_raw['forum'] == forum).sum() for forum in posts_raw['forum'].unique()}

    p = posts_raw.copy()

    # Filtro 1: texto nulo o vacío
    p = p[p['pagetext'].notna() & (p['pagetext'].astype(str).str.strip() != '')]

    # Filtro 2: duplicados por (forum, postid)
    p = p.drop_duplicates(subset=['forum', 'postid'])

    # Filtro 3: epoch-0 (timestamps inválidos anteriores al año 2000)
    p['dateline'] = pd.to_datetime(p['dateline'], errors='coerce', utc=True)
    p = p[p['dateline'].notna() & (p['dateline'].dt.year >= 2000)]

    # Filtro 4: texto demasiado corto
    p = p[p['pagetext'].astype(str).str.len() >= 10]

    posts_clean = p.reset_index(drop=True)

    print(f"{'Foro':<35} {'Antes':>10} {'Después':>10} {'Retenidos':>10}")
    print('-' * 70)
    for forum in sorted(counts_before):
        before = counts_before[forum]
        after  = (posts_clean['forum'] == forum).sum()
        pct    = after / before * 100 if before > 0 else 0
        print(f"{forum:<35} {before:>10,} {after:>10,} {pct:>9.1f}%")
    print('-' * 70)
    print(f"{'TOTAL':<35} {len(posts_raw):>10,} {len(posts_clean):>10,} "
          f"{len(posts_clean)/len(posts_raw)*100:>9.1f}%")

## Sección 3 — Limpieza de usuarios

Para los usuarios aplicamos una limpieza similar pero con pasos específicos para identidades.

### Normalización de handles: ¿por qué lowercase + strip?

El mismo usuario puede haberse registrado como `"Hacker99"`, `"hacker99"`, o `" hacker99 "` (con espacio).  
Para el pivoting cross-foro, queremos que los tres se traten como el mismo handle.

- `str.lower()` convierte todo a minúsculas
- `str.strip()` elimina espacios al inicio y al final

Guardamos el username original en `username_raw` por si necesitamos recuperar la capitalización  
original para mostrarla en resultados.

### Prefijado de userids

Cada foro numera sus usuarios independientemente: el usuario con `userid = 42` en OGUsers  
es una persona completamente distinta al usuario `userid = 42` en Cracked.to.  
Si concatenamos todos los foros sin prefijo, tenemos **colisiones de IDs**.

La solución es prefijar: `OGUsers_2019_42`, `Cracked.to_2019.01_42`.  
Así cada userid es globalmente único en nuestro corpus combinado.

In [ ]:
if len(users_raw) == 0:
    print("Sin usuarios para limpiar.")
    users_clean = users_raw.copy()
else:
    ucounts_before = {forum: (users_raw['forum'] == forum).sum() for forum in users_raw['forum'].unique()}

    u = users_raw.copy()

    # 1. Duplicados por (forum, userid)
    u = u.drop_duplicates(subset=['forum', 'userid'])

    # 2. Username nulo
    u = u[u['username'].notna() & (u['username'].astype(str).str.strip() != '')]

    # 3. Normalización: guardar original, normalizar para comparación
    u['username_raw'] = u['username']
    u['username']     = u['username'].astype(str).str.lower().str.strip()

    # 4. Epoch-0 en joindate (fecha de registro inválida)
    u['joindate'] = pd.to_datetime(u['joindate'], errors='coerce', utc=True)
    epoch_mask = u['joindate'].notna() & (u['joindate'].dt.year < 2000)
    u.loc[epoch_mask, 'joindate'] = pd.NaT

    # 5. Prefijado de userid para evitar colisiones cross-foro
    u['userid_global'] = u['forum'] + '_' + u['userid'].astype(str)

    users_clean = u.reset_index(drop=True)

    print(f"{'Foro':<35} {'Antes':>10} {'Después':>10} {'Retenidos':>10}")
    print('-' * 70)
    for forum in sorted(ucounts_before):
        before = ucounts_before[forum]
        after  = (users_clean['forum'] == forum).sum()
        pct    = after / before * 100 if before > 0 else 0
        print(f"{forum:<35} {before:>10,} {after:>10,} {pct:>9.1f}%")
    print('-' * 70)
    print(f"{'TOTAL':<35} {len(users_raw):>10,} {len(users_clean):>10,} "
          f"{len(users_clean)/len(users_raw)*100:>9.1f}%")

## Sección 4 — Preparación para análisis estructural

El análisis de redes (notebook 02) necesita saber qué usuarios participaron en qué threads.  
Construimos una tabla de co-participación: `(userid_global, threadid, forum)` por cada post.

También prefijamos el `threadid` para evitar colisiones entre foros  
(el thread `42` de OGUsers ≠ el thread `42` de RaidForums).

In [ ]:
if len(posts_clean) > 0 and len(users_clean) > 0:
    # Tabla de participación usuario ↔ thread con IDs globales
    participation = posts_clean[['forum', 'userid', 'threadid']].copy()
    participation['userid_global']  = participation['forum'] + '_' + participation['userid'].astype(str)
    participation['threadid_global'] = participation['forum'] + '_' + participation['threadid'].astype(str)
    participation = participation.drop_duplicates(subset=['userid_global', 'threadid_global'])

    # Guardar para el notebook 02
    part_path = HF_RESULTS / 'participation_user_thread.parquet'
    participation.to_parquet(part_path, index=False)

    print(f"Tabla de participación: {len(participation):,} filas (usuario ↔ thread)")
    print(f"Guardada en: {part_path}")
    print(f"\nEjemplo de 5 filas:")
    print(participation.head(5).to_string(index=False))
else:
    print("Sin datos suficientes para construir tabla de participación.")

## Sección 5 — Preparación para análisis semántico

El análisis semántico (notebook 03) trabaja con **el texto de cada usuario**,  
no con posts individuales. Necesitamos construir el **corpus por usuario**:
todos los posts de un usuario concatenados en un solo documento.

### Estrategia: top-N posts más largos

Un usuario activo puede tener miles de posts. Usar todos sería muy lento para embeddings.

La estrategia elegida es tomar los **50 posts más largos** por usuario.  
¿Por qué los más largos? Porque los posts cortos (`+1`, `thanks`, `nice`) son ruido semántico:  
no revelan el estilo ni los temas del usuario. Los posts largos tienen contexto real.

Esta tabla (corpus_by_user) es el input directo del notebook 03.

In [ ]:
MAX_POSTS_PER_USER = 50

if len(posts_clean) > 0:
    posts_clean['text_len'] = posts_clean['pagetext'].astype(str).str.len()

    # Para cada usuario, tomar los MAX_POSTS_PER_USER más largos
    corpus_rows = []
    for (forum, userid), grp in posts_clean.groupby(['forum', 'userid']):
        top_posts = grp.nlargest(MAX_POSTS_PER_USER, 'text_len')
        combined_text = ' '.join(top_posts['pagetext'].astype(str).tolist())
        corpus_rows.append({
            'forum':         forum,
            'userid':        userid,
            'userid_global': forum + '_' + str(userid),
            'n_posts':       len(grp),
            'n_posts_used':  len(top_posts),
            'total_chars':   len(combined_text),
            'corpus_text':   combined_text,
        })

    corpus_df = pd.DataFrame(corpus_rows)

    # Guardar corpus para notebook 03
    corpus_path = HF_RESULTS / 'corpus_by_user.parquet'
    corpus_df.drop(columns=['corpus_text']).to_parquet(corpus_path, index=False)

    print(f"Corpus por usuario: {len(corpus_df):,} usuarios")
    print(f"Columnas: {list(corpus_df.columns)}")
    print(f"\nDistribución de posts por usuario por foro:")
    print(corpus_df.groupby('forum')['n_posts'].describe().round(1).to_string())
    print(f"\nGuardado metadatos en: {corpus_path}")
else:
    print("Sin posts limpios para construir corpus por usuario.")

## Sección 6 — Detección de idioma definitiva

Aquí hacemos la detección de idioma sobre el corpus limpio (no los datos crudos del notebook 00).  

La diferencia con el notebook 00 es que ahora trabajamos sobre `posts_clean`:  
posts sin HTML, sin duplicados, con texto mínimo de 10 caracteres.  
Eso mejora la precisión de `langdetect`.

El resultado de esta celda se guarda como `lang_route` en cada post:  
`'en'` para foros en inglés, `'ru'` para Exploit.in.  
Esta columna es la que usan los notebooks posteriores para decidir qué pipeline aplicar.

In [ ]:
import re
import random
from langdetect import detect, LangDetectException

SAMPLE_PER_FORUM = 500

def strip_html(text):
    return re.sub(r"<[^>]+>", " ", str(text or ""))

forum_lang_decision = {}

if len(posts_clean) > 0:
    for forum, grp in posts_clean.groupby('forum'):
        texts  = grp['pagetext'].dropna().astype(str).tolist()
        random.seed(42)
        sample = random.sample(texts, min(SAMPLE_PER_FORUM, len(texts)))

        lang_counts = {}
        for raw in sample:
            text = strip_html(raw)[:2000].strip()
            if len(text) < 30:
                continue
            try:
                lang = detect(text)
            except LangDetectException:
                lang = 'unknown'
            lang_counts[lang] = lang_counts.get(lang, 0) + 1

        if lang_counts:
            total    = sum(lang_counts.values())
            top_lang = max(lang_counts, key=lang_counts.get)
            top_pct  = lang_counts[top_lang] / total * 100
            route    = 'ru' if top_lang == 'ru' else 'en'
            forum_lang_decision[forum] = route
            flag = '⚠️  ruso → pipeline separado' if route == 'ru' else '✓  inglés'
            print(f"  {forum.split('_')[0]:<18} → {top_lang} ({top_pct:.0f}%)  {flag}")

    # Añadir columna lang_route al DataFrame de posts
    posts_clean['lang_route'] = posts_clean['forum'].map(forum_lang_decision).fillna('en')

    print(f"\nDecisión por foro: {forum_lang_decision}")
    print(f"\nResumen lang_route:")
    print(posts_clean['lang_route'].value_counts().to_string())
else:
    print("Sin posts para detección de idioma.")

## Sección 7 — Guardar outputs finales

Guardamos los tres artefactos que consumen los notebooks siguientes:

| Archivo | Descripción | Usado en |
|---------|-------------|----------|
| `posts_clean.parquet` | Todos los posts limpios con `forum` y `lang_route` | 02, 03 |
| `users_clean.parquet` | Todos los usuarios normalizados con `userid_global` | 02, 03 |
| `participation_user_thread.parquet` | Tabla (userid_global, threadid_global) | 02 |

**Parquet** es un formato de archivo columnar: almacena los datos de forma comprimida  
y es mucho más rápido de cargar que CSV para DataFrames grandes (millones de filas).

In [ ]:
if len(posts_clean) > 0:
    posts_out = HF_RESULTS / 'posts_clean.parquet'
    posts_clean.to_parquet(posts_out, index=False)
    print(f"posts_clean.parquet → {len(posts_clean):,} filas ({posts_out.stat().st_size / 1e6:.1f} MB)")

if len(users_clean) > 0:
    users_out = HF_RESULTS / 'users_clean.parquet'
    users_clean.to_parquet(users_out, index=False)
    print(f"users_clean.parquet → {len(users_clean):,} filas ({users_out.stat().st_size / 1e6:.1f} MB)")

print("\n=== Validación rápida ===")
if (HF_RESULTS / 'posts_clean.parquet').exists():
    check_posts = pd.read_parquet(HF_RESULTS / 'posts_clean.parquet')
    assert check_posts['pagetext'].isna().sum() == 0, "Hay posts con texto nulo!"
    assert (check_posts['dateline'].dt.year < 2000).sum() == 0, "Hay timestamps epoch-0!"
    print(f"posts_clean: {len(check_posts):,} filas — OK")

if (HF_RESULTS / 'users_clean.parquet').exists():
    check_users = pd.read_parquet(HF_RESULTS / 'users_clean.parquet')
    assert check_users['username'].isna().sum() == 0, "Hay usuarios sin username!"
    print(f"users_clean: {len(check_users):,} filas — OK")

print("\nTodas las validaciones pasaron. El corpus está listo para el análisis.")

## Resumen de lo que hicimos

En este notebook:

1. **Cargamos** los 5 foros con `load_forum()`, normalizando columnas a un schema común.
2. **Limpiamos posts**: eliminamos nulos, duplicados, timestamps inválidos y texto demasiado corto.
3. **Limpiamos usuarios**: normalizamos handles, eliminamos duplicados y prefijamos userids.
4. **Construimos la tabla de participación** usuario↔thread para análisis de redes.
5. **Construimos el corpus por usuario** (top-50 posts más largos) para embeddings.
6. **Detectamos el idioma** definitivo por foro y lo guardamos como `lang_route`.
7. **Guardamos** los tres archivos Parquet de output.

**Siguiente paso**: notebook `02_analisis_estructural.ipynb` — redes de co-participación, pivoting y TF-IDF.